# mariadb smoke test

Building the `mariadb` package needs MariaDB Connector/C (`mariadb_config` on `PATH`, e.g. from the `libmariadb-dev` package); no server is needed for these offline checks.

## Example 1: Module version and DB-API connection interface

In [ ]:
# Example 1: Module version and DB-API connection interface
import inspect
import mariadb

print("🔌 MariaDB Connector/Python - module version and connection interface (no server needed)")

# The package reports its own version as a string and as a tuple of integers
assert isinstance(mariadb.__version__, str) and mariadb.__version__, mariadb.__version__
assert all(isinstance(part, int) for part in mariadb.__version_info__[:3]), mariadb.__version_info__
print(f"   • mariadb {mariadb.__version__} {tuple(mariadb.__version_info__)}")

# connect() is the DB-API entry point; it takes the connection settings as keyword arguments
assert callable(mariadb.connect)
connect_params = inspect.signature(mariadb.connect).parameters
assert any(p.kind is inspect.Parameter.VAR_KEYWORD for p in connect_params.values()), connect_params
print(f"   • mariadb.connect{inspect.signature(mariadb.connect)}")

# Connection and Cursor classes expose the DB-API 2.0 methods
connection_methods = ["cursor", "commit", "rollback", "close", "ping", "begin"]
cursor_methods = ["execute", "executemany", "fetchone", "fetchmany", "fetchall", "close", "callproc"]
for cls, methods in ((mariadb.Connection, connection_methods), (mariadb.Cursor, cursor_methods)):
    missing = [name for name in methods if not callable(getattr(cls, name, None))]
    assert not missing, (cls.__name__, missing)
    print(f"   • {cls.__module__}.{cls.__name__}: {', '.join(methods)}")
for attribute in ("description", "rowcount", "lastrowid", "arraysize"):
    assert hasattr(mariadb.Cursor, attribute), attribute

print("✅ Module version and connection interface checked")

## Example 2: DB-API type constructors and type objects

In [ ]:
# Example 2: DB-API type constructors and type objects
import datetime
import mariadb
from mariadb.constants import FIELD_TYPE

print("📋 MariaDB DB-API type constructors and type objects")

# The constructors build the Python values that the connector binds to ? placeholders
assert mariadb.Date(2024, 1, 2) == datetime.date(2024, 1, 2)
assert mariadb.Time(13, 5, 9) == datetime.time(13, 5, 9)
assert mariadb.Timestamp(2024, 1, 2, 3, 4, 5) == datetime.datetime(2024, 1, 2, 3, 4, 5)
assert mariadb.Binary(b"\x00\x01") == b"\x00\x01"
ticks = datetime.datetime(2024, 1, 2, 3, 4, 5).timestamp()  # local time, as the *FromTicks constructors use
assert mariadb.DateFromTicks(ticks) == datetime.date(2024, 1, 2)
assert mariadb.TimestampFromTicks(ticks) == datetime.datetime(2024, 1, 2, 3, 4, 5)
print("   • Date, Time, Timestamp, Binary and the *FromTicks variants build datetime and bytes values")

# Each type object compares equal to every column type code in its group (cursor.description reports these codes)
groups = {
    "NUMBER": (mariadb.NUMBER, [FIELD_TYPE.TINY, FIELD_TYPE.LONG, FIELD_TYPE.LONGLONG, FIELD_TYPE.DOUBLE, FIELD_TYPE.NEWDECIMAL]),
    "STRING": (mariadb.STRING, [FIELD_TYPE.VARCHAR, FIELD_TYPE.VAR_STRING, FIELD_TYPE.STRING]),
    "DATETIME": (mariadb.DATETIME, [FIELD_TYPE.DATETIME, FIELD_TYPE.TIMESTAMP]),
    "BINARY": (mariadb.BINARY, [FIELD_TYPE.BLOB, FIELD_TYPE.LONG_BLOB]),
}
for name, (type_object, codes) in groups.items():
    assert all(type_object == code for code in codes), (name, codes)
    print(f"   • mariadb.{name} matches column type codes {codes}")
assert mariadb.NUMBER != FIELD_TYPE.VARCHAR and mariadb.STRING != FIELD_TYPE.LONG

print("✅ Type constructors and type objects checked")

## Example 3: Exception hierarchy

In [ ]:
# Example 3: Exception hierarchy
import mariadb

print("🧭 MariaDB exception hierarchy")

# Every error the connector raises derives from mariadb.Error, so one except clause can catch them all
error_classes = ["InterfaceError", "DatabaseError", "DataError", "OperationalError", "IntegrityError",
                 "InternalError", "ProgrammingError", "NotSupportedError", "PoolError"]
for name in error_classes:
    cls = getattr(mariadb, name)
    assert issubclass(cls, mariadb.Error), (name, cls.__mro__)
    print(f"   • mariadb.{name} is a mariadb.Error")
assert issubclass(mariadb.Error, Exception) and issubclass(mariadb.Warning, Exception)

# The exceptions carry the message they were raised with
try:
    raise mariadb.IntegrityError("Duplicate entry 'alice' for key 'username'")
except mariadb.Error as e:
    assert type(e) is mariadb.IntegrityError and "Duplicate entry" in str(e), (type(e), str(e))
    print(f"   • caught {type(e).__name__} through mariadb.Error: {e}")

print("✅ Exception hierarchy checked")

## Example 4: Connection failures and pool validation without a server

In [ ]:
# Example 4: Connection failures and pool validation without a server
import socket
import time
import mariadb

print("⚡ MariaDB connection failures and connection-pool validation (no server needed)")

# Reserve a free loopback port and release it again, so that nothing is listening there
with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    closed_port = probe.getsockname()[1]

start = time.perf_counter()
try:
    mariadb.connect(host="127.0.0.1", port=closed_port, user="example", connect_timeout=1)
except mariadb.OperationalError as e:
    elapsed = time.perf_counter() - start
    print(f"   • connect() to closed port {closed_port}: {type(e).__name__}, errno {e.errno}, sqlstate {e.sqlstate} ({elapsed:.3f}s)")
    assert e.errno in (2002, 2003), e.errno  # client errors CR_CONNECTION_ERROR / CR_CONN_HOST_ERROR
else:
    raise AssertionError(f"connect() to closed port {closed_port} should fail")

# The pool size is validated before any connection is attempted
try:
    mariadb.ConnectionPool(pool_name="oversized_pool", pool_size=mariadb.MAX_POOL_SIZE + 1)
except mariadb.ProgrammingError as e:
    print(f"   • pool_size={mariadb.MAX_POOL_SIZE + 1}: {type(e).__name__}: {e}")
else:
    raise AssertionError("a pool larger than MAX_POOL_SIZE should be rejected")

# A pool created without connection arguments holds no connections, so get_connection() raises PoolError
pool = mariadb.ConnectionPool(pool_name="empty_pool", pool_size=2)
try:
    assert (pool.pool_name, pool.pool_size, pool.connection_count) == ("empty_pool", 2, 0)
    try:
        pool.get_connection()
    except mariadb.PoolError as e:
        print(f"   • get_connection() from an empty pool: {type(e).__name__}: {e}")
    else:
        raise AssertionError("an empty pool should not hand out a connection")
finally:
    pool.close()

print("✅ Connection failures and pool validation checked")

## Example 5: DB-API module interface and protocol constants

In [ ]:
# Example 5: DB-API module interface and protocol constants
import mariadb
from mariadb.constants import CLIENT, FIELD_TYPE, INDICATOR

print("🏭 MariaDB DB-API module interface and protocol constants")

# DB-API 2.0 module globals: qmark placeholders (?), connections may be shared between threads
assert mariadb.apilevel == "2.0" and mariadb.paramstyle == "qmark", (mariadb.apilevel, mariadb.paramstyle)
assert int(mariadb.threadsafety) >= 1, mariadb.threadsafety
assert all(isinstance(part, int) for part in mariadb.client_version_info), mariadb.client_version_info
print(f"   • mariadb {mariadb.__version__} on MariaDB Connector/C {'.'.join(map(str, mariadb.client_version_info))}")
print(f"   • apilevel {mariadb.apilevel}, paramstyle {mariadb.paramstyle!r}, threadsafety {mariadb.threadsafety}")

# Column type codes are the protocol values that cursor.description reports
assert (FIELD_TYPE.TINY, FIELD_TYPE.LONG, FIELD_TYPE.VARCHAR, FIELD_TYPE.JSON, FIELD_TYPE.BLOB) == (1, 3, 15, 245, 252)

# Indicators stand in for a value in executemany() rows: NULL, the column DEFAULT, IGNORE the column, or IGNORE_ROW
indicators = [INDICATOR.NULL, INDICATOR.DEFAULT, INDICATOR.IGNORE, INDICATOR.IGNORE_ROW]
assert len({id(indicator) for indicator in indicators}) == 4, indicators
print(f"   • executemany() indicators: NULL, DEFAULT, IGNORE, IGNORE_ROW")

# CLIENT capability flags are single bits that can be OR-ed into connect(client_flag=...)
flags = {"FOUND_ROWS": CLIENT.FOUND_ROWS, "COMPRESS": CLIENT.COMPRESS, "MULTI_STATEMENTS": CLIENT.MULTI_STATEMENTS}
for name, flag in flags.items():
    assert flag > 0 and flag & (flag - 1) == 0, (name, flag)
combined = CLIENT.FOUND_ROWS | CLIENT.MULTI_STATEMENTS
assert combined & CLIENT.MULTI_STATEMENTS and not combined & CLIENT.COMPRESS
print(f"   • client flags {flags}, combined FOUND_ROWS | MULTI_STATEMENTS = {combined}")

print("✅ Module interface and constants checked")